# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamKottish/FlyRankML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token: "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MAR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-03/*.parquet')"
)

FACT_APR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-04/*.parquet')"
)

print("✓ Connected")
print("Main audit window: March 2026")
print("Retrospective check: April 2026")

Paste your Hugging Face READ token: ··········
✓ Connected
Main audit window: March 2026
Retrospective check: April 2026


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*


Before testing any CTR assumptions, I first inspect the distributions of the main March search signals.

The unit used here is one pseudonymized client-content item. I examine impressions, clicks, CTR, average position, and position volatility. Search traffic is expected to be heavy-tailed, meaning a relatively small number of pages may account for very large numbers of impressions or clicks.

Because heavy tails can make ordinary averages and Pearson correlations misleading, the later signal tests use minimum-volume floors, grouped summaries, weighted CTR, and Spearman rank correlations where appropriate. CTR is always interpreted together with its impression denominator.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build one March row per client + content item

march_pages = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS ctr,

        SUM(
            CASE
                WHEN gsc_avg_position > 0 AND gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_avg_position > 0 AND gsc_impressions > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS avg_position,

        STDDEV_SAMP(
            CASE
                WHEN gsc_avg_position > 0 AND gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS position_std,

        COUNT(
            DISTINCT CASE
                WHEN gsc_impressions > 0
                THEN report_date
            END
        ) AS active_days

    FROM {FACT_MAR}

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) > 0
""").df()

march_pages["position_std"] = (
    march_pages["position_std"].fillna(0)
)

print(f"March content items with impressions: {len(march_pages):,}")

# Distribution summary
fields = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_std",
]

summary = march_pages[fields].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
).T[
    ["count", "mean", "50%", "90%", "95%", "99%", "max"]
].round(2)

display(summary)

# Heavy-tail check
tail_check = pd.DataFrame({
    "raw_skew": [
        march_pages["impressions"].skew(),
        march_pages["clicks"].skew()
    ],
    "log1p_skew": [
        np.log1p(march_pages["impressions"]).skew(),
        np.log1p(march_pages["clicks"]).skew()
    ]
}, index=["impressions", "clicks"]).round(2)

print("\nHeavy-tail check:")
display(tail_check)

print(
    "If raw skew is much larger than log1p skew, "
    "the traffic metric is strongly right-skewed."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March content items with impressions: 176,738


,count,mean,50%,90%,95%,99%,max
impressions,176738.0,1587.99,173.00,3930.00,7238.15,21799.78,617124.00
clicks,176738.0,4.65,0.00,10.00,22.00,73.00,5668.00
ctr,176738.0,0.46,0.00,0.62,1.09,5.88,100.00
avg_position,175304.0,16.72,8.57,43.08,60.25,81.34,309.00
position_std,176738.0,8.49,4.64,22.45,28.39,40.14,238.29



Heavy-tail check:


,raw_skew,log1p_skew
impressions,19.40,-0.05
clicks,73.41,1.76


If raw skew is much larger than log1p skew, the traffic metric is strongly right-skewed.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal 1 — Search position and CTR:** The claim is that pages with better search positions tend to capture a larger share of their impressions as clicks. I test this only on pages with at least 500 impressions to reduce low-volume CTR noise.

**Signal 2 — Exposure and clicks:** The claim is that pages with more observed impressions generally receive more observed clicks. I use a Spearman correlation because both variables are strongly heavy-tailed.

**Signal 3 — Position volatility and CTR underperformance:** The claim is that pages whose search positions move around more may have less stable or weaker CTR relative to pages at similar average positions. This signal is less certain, so a MIXED or FALSE result is acceptable.

Each test reports its sample size and returns a verdict of CONFIRMED, OPPOSITE, MIXED, or FALSE rather than forcing a positive result.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
MIN_IMPRESSIONS = 500
MIN_N = 50

audit = march_pages[
    (march_pages["impressions"] >= MIN_IMPRESSIONS)
    & march_pages["avg_position"].notna()
].copy()


def position_band(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    return "deep"


audit["position_band"] = (
    audit["avg_position"].apply(position_band)
)


# ======================================================
# SIGNAL 1
# Better position -> higher CTR
# ======================================================

rho1 = audit[
    ["avg_position", "ctr"]
].corr(method="spearman").iloc[0, 1]

band_table = (
    audit.groupby("position_band")
    .agg(
        n=("content_hash_id", "size"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        median_position=("avg_position", "median"),
    )
)

band_table["weighted_ctr"] = (
    100 * band_table["clicks"]
    / band_table["impressions"]
)

band_order = [
    "top_3",
    "page_1",
    "striking",
    "page_3_5",
    "deep",
]

band_table = band_table.reindex(
    [x for x in band_order if x in band_table.index]
)

print("SIGNAL 1 — Better position -> higher CTR")
display(band_table.round(3))

if len(audit) < MIN_N:
    verdict1 = "INSUFFICIENT DATA"
elif rho1 <= -0.20:
    verdict1 = "CONFIRMED"
elif rho1 >= 0.20:
    verdict1 = "OPPOSITE"
elif abs(rho1) < 0.05:
    verdict1 = "FALSE"
else:
    verdict1 = "MIXED"

print(f"Spearman rho: {rho1:.3f}")
print("VERDICT 1:", verdict1)


# ======================================================
# SIGNAL 2
# More impressions -> more clicks
# ======================================================

signal2_df = march_pages[
    march_pages["impressions"] >= 100
].copy()

rho2 = signal2_df[
    ["impressions", "clicks"]
].corr(method="spearman").iloc[0, 1]

signal2_df["impression_quartile"] = pd.qcut(
    signal2_df["impressions"],
    q=4,
    duplicates="drop"
)

exposure_table = (
    signal2_df.groupby(
        "impression_quartile",
        observed=True
    )
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median"),
        median_clicks=("clicks", "median"),
        total_impressions=("impressions", "sum"),
        total_clicks=("clicks", "sum"),
    )
)

print("\nSIGNAL 2 — More exposure -> more clicks")
display(exposure_table)

if len(signal2_df) < MIN_N:
    verdict2 = "INSUFFICIENT DATA"
elif rho2 >= 0.50:
    verdict2 = "CONFIRMED"
elif rho2 <= -0.20:
    verdict2 = "OPPOSITE"
elif abs(rho2) < 0.10:
    verdict2 = "FALSE"
else:
    verdict2 = "MIXED"

print(f"Spearman rho: {rho2:.3f}")
print("VERDICT 2:", verdict2)


# ======================================================
# SIGNAL 3
# Position volatility -> CTR underperformance
# ======================================================

audit["band_median_ctr"] = (
    audit.groupby("position_band")["ctr"]
    .transform("median")
)

audit["ctr_gap_pp"] = (
    audit["band_median_ctr"]
    - audit["ctr"]
)

rho3 = audit[
    ["position_std", "ctr_gap_pp"]
].corr(method="spearman").iloc[0, 1]


def volatility_band(std):
    if std < 5:
        return "low"
    elif std < 10:
        return "medium"
    return "high"


audit["volatility_band"] = (
    audit["position_std"].apply(volatility_band)
)

volatility_table = (
    audit.groupby("volatility_band")
    .agg(
        n=("content_hash_id", "size"),
        median_position_std=("position_std", "median"),
        median_ctr_gap_pp=("ctr_gap_pp", "median"),
        median_ctr=("ctr", "median"),
    )
)

print(
    "\nSIGNAL 3 — Position volatility -> CTR underperformance"
)
display(volatility_table.round(3))

large_enough_buckets = (
    volatility_table["n"] >= MIN_N
).sum()

if len(audit) < MIN_N or large_enough_buckets < 2:
    verdict3 = "INSUFFICIENT DATA"
elif rho3 >= 0.10:
    verdict3 = "CONFIRMED"
elif rho3 <= -0.10:
    verdict3 = "OPPOSITE"
elif abs(rho3) < 0.03:
    verdict3 = "FALSE"
else:
    verdict3 = "MIXED"

print(f"Spearman rho: {rho3:.3f}")
print("VERDICT 3:", verdict3)


print("\nFINAL SIGNAL VERDICTS")
print("Signal 1:", verdict1)
print("Signal 2:", verdict2)
print("Signal 3:", verdict3)

SIGNAL 1 — Better position -> higher CTR


,n,impressions,clicks,median_position,weighted_ctr
position_band,,,,,
top_3,7891,40673101.0,157985.0,2.261,0.388
page_1,32326,143358732.0,464830.0,5.535,0.324
striking,10552,28561367.0,92651.0,13.902,0.324
page_3_5,10471,54978553.0,76830.0,28.490,0.140
deep,684,1344366.0,453.0,60.071,0.034


Spearman rho: -0.265
VERDICT 1: CONFIRMED

SIGNAL 2 — More exposure -> more clicks


,n,median_impressions,median_clicks,total_impressions,total_clicks
impression_quartile,,,,,
"(99.999, 283.0]",25370,171.0,0.0,4494744.0,10222.0
"(283.0, 786.0]",25351,471.0,0.0,12450892.0,29781.0
"(786.0, 2514.0]",25360,1360.0,2.0,36880154.0,104574.0
"(2514.0, 617124.0]",25360,5239.0,13.0,224970293.0,670798.0


Spearman rho: 0.740
VERDICT 2: CONFIRMED

SIGNAL 3 — Position volatility -> CTR underperformance


,n,median_position_std,median_ctr_gap_pp,median_ctr
volatility_band,,,,
high,5525,12.680,0.051,0.056
low,41268,2.006,-0.018,0.221
medium,15131,6.762,0.014,0.124


Spearman rho: 0.116
VERDICT 3: CONFIRMED

FINAL SIGNAL VERDICTS
Signal 1: CONFIRMED
Signal 2: CONFIRMED
Signal 3: CONFIRMED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*


I test the assumption behind FlyRank's CTR-review family of flags: a page that already has meaningful search visibility and a reasonably strong position, but captures unusually few clicks, may deserve human review.

I do not use FlyRank's private `needs_ctr_fix` flag itself. Instead, I use the public starter baseline's transparent `low_ctr_visible_page` condition as a testable analog:

- at least 500 March impressions,
- average March position between 1 and 20,
- March CTR below 0.5%.

I then ask whether those March candidates are more likely than comparable non-candidates to remain position-adjusted CTR underperformers in April.

April is used only as a later observed test period. It does not enter the March flag-like condition.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ------------------------------------------------------
# March flag-like population
# ------------------------------------------------------

march_flag_test = march_pages[
    (march_pages["impressions"] >= 500)
    & (march_pages["avg_position"] > 0)
    & (march_pages["avg_position"] <= 20)
].copy()

march_flag_test["flag_like_ctr_candidate"] = (
    march_flag_test["ctr"] < 0.5
)


# ------------------------------------------------------
# April outcome
# ------------------------------------------------------

april_pages = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS april_ctr,

        SUM(
            CASE
                WHEN gsc_avg_position > 0 AND gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_avg_position > 0 AND gsc_impressions > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS april_avg_position

    FROM {FACT_APR}

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 500
""").df()

april_pages = april_pages[
    (april_pages["april_avg_position"] > 0)
    & (april_pages["april_avg_position"] <= 20)
].copy()

april_pages["april_position_band"] = (
    april_pages["april_avg_position"]
    .apply(position_band)
)

april_pages["april_band_median_ctr"] = (
    april_pages
    .groupby("april_position_band")["april_ctr"]
    .transform("median")
)

april_pages["april_ctr_gap_pp"] = (
    april_pages["april_band_median_ctr"]
    - april_pages["april_ctr"]
)

# Later observed opportunity definition
april_pages["april_underperformer"] = (
    april_pages["april_ctr_gap_pp"] > 0.10
)


# ------------------------------------------------------
# Join March condition to April result
# ------------------------------------------------------

flag_eval = march_flag_test.merge(
    april_pages[
        [
            "client_hash_id",
            "content_hash_id",
            "april_impressions",
            "april_ctr",
            "april_avg_position",
            "april_ctr_gap_pp",
            "april_underperformer",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

flag_summary = (
    flag_eval.groupby("flag_like_ctr_candidate")
    .agg(
        n=("content_hash_id", "size"),
        april_underperformance_rate=(
            "april_underperformer",
            "mean"
        ),
        median_april_gap_pp=(
            "april_ctr_gap_pp",
            "median"
        ),
        median_april_ctr=(
            "april_ctr",
            "median"
        ),
    )
)

print("Flag-linked test:")
display(flag_summary.round(3))


# ------------------------------------------------------
# Verdict
# ------------------------------------------------------

if (
    True not in flag_summary.index
    or False not in flag_summary.index
):
    flag_verdict = "INSUFFICIENT DATA"

elif (
    flag_summary.loc[True, "n"] < 50
    or flag_summary.loc[False, "n"] < 50
):
    flag_verdict = "INSUFFICIENT DATA"

else:
    flagged_rate = (
        flag_summary.loc[
            True,
            "april_underperformance_rate"
        ]
    )

    comparison_rate = (
        flag_summary.loc[
            False,
            "april_underperformance_rate"
        ]
    )

    difference = flagged_rate - comparison_rate

    if difference >= 0.05:
        flag_verdict = "CONFIRMED"
    elif difference <= -0.05:
        flag_verdict = "OPPOSITE"
    elif abs(difference) < 0.02:
        flag_verdict = "FALSE"
    else:
        flag_verdict = "MIXED"

    print(
        f"Flag-like candidate April rate: "
        f"{flagged_rate:.1%}"
    )

    print(
        f"Comparison April rate: "
        f"{comparison_rate:.1%}"
    )

    print(
        f"Difference: "
        f"{difference:+.1%}"
    )

print("\nFLAG-LINKED VERDICT:", flag_verdict)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Flag-linked test:


,n,april_underperformance_rate,median_april_gap_pp,median_april_ctr
flag_like_ctr_candidate,,,,
False,8433,0.037,-0.348,0.547
True,30912,0.379,0.055,0.142


Flag-like candidate April rate: 37.9%
Comparison April rate: 3.7%
Difference: +34.2%

FLAG-LINKED VERDICT: CONFIRMED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*


A content team should not judge CTR using one universal threshold. CTR needs to be interpreted with search position and enough impression volume, because low-volume rates are noisy and ranking position strongly affects click opportunity.

Signals that are CONFIRMED can remain useful inputs for prioritizing human review, while MIXED, FALSE, or OPPOSITE signals should be weakened or removed rather than treated as facts. The flag-linked result is observational decision-support only; it does not prove that changing a title, snippet, or page will cause CTR to improve.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
verdict_summary = pd.DataFrame({
    "test": [
        "Position vs CTR",
        "Exposure vs clicks",
        "Position volatility vs CTR gap",
        "CTR flag-linked persistence"
    ],
    "verdict": [
        verdict1,
        verdict2,
        verdict3,
        flag_verdict
    ]
})

display(verdict_summary)

print("\nPractical rule:")
print(
    "Keep CONFIRMED signals, investigate MIXED signals, "
    "and do not promote FALSE or OPPOSITE signals into "
    "positive ranking rules without stronger evidence."
)

,test,verdict
0,Position vs CTR,CONFIRMED
1,Exposure vs clicks,CONFIRMED
2,Position volatility vs CTR gap,CONFIRMED
3,CTR flag-linked persistence,CONFIRMED



Practical rule:
Keep CONFIRMED signals, investigate MIXED signals, and do not promote FALSE or OPPOSITE signals into positive ranking rules without stronger evidence.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.